The user asks a question. Relevant information is retrieved from the vector database and passed to the LLM as context, which then generates the answer.

In [14]:
# pip install langchain-groq python-dotenv

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.load_local(
    "../data/faiss_index", embedding_model,
    allow_dangerous_deserialization=True
)
print("Loaded chunks:", vector_store.index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded chunks: 680


In [16]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv("../.env")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


In [17]:
THRESHOLD = 0.45

def ask(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=3)
    best_score = results[0][1]
    print("Best score:", round(best_score, 2))

    if best_score < THRESHOLD:
        print("Answer: I don't have this information in the documents.")
        print("Action: Escalate to human (low confidence)")
        return

    docs = [doc for doc, score in results]
    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    print("\nAnswer:", response.content)
    print("\nSources:")
    seen = set()
    for d in docs:
        if d.metadata["source"] not in seen:
            seen.add(d.metadata["source"])
            print("-", d.metadata["title"], "|", d.metadata["source"])


In [19]:
ask("How do I share a page with a guest?")
ask("Who is the CEO of Notion?")
ask("How do I change my Notion plan?")
ask("How do I set a reminder in Notion?")
ask("What is the weather in Delhi today?")








# def check_scores(question):
#     results = vector_store.similarity_search_with_relevance_scores(question, k=3)
#     print("\nQuestion:", question)
#     for doc, score in results:
#         print(f"{score:.2f} | {doc.metadata['title']}")

# check_scores("How do I share a page with a guest?")
# check_scores("Who is the CEO of Notion?")
# check_scores("How do I change my Notion plan?")



Best score: 0.67

Answer: 1. Click **Share** at the top of the page.  
2. In the sharing dialog, enter the guest’s email address (or select their name if they’re already in the workspace).  
3. Click the dropdown next to their name and choose the desired permission level.  
4. Press **Invite**.  

The guest will receive an invitation and, once they accept, will have the permissions you set. (If the email belongs to one of your organization’s domains, the person will be added as a member instead of a guest.)

Sources:
- Sharing & permissions settings in Notion | https://www.notion.com/en-gb/help/sharing-and-permissions
- Intro to workspaces in Notion | https://www.notion.com/help/intro-to-workspaces
Best score: 0.37
Answer: I don't have this information in the documents.
Action: Escalate to human (low confidence)
Best score: 0.65

Answer: To change your Notion plan:

1. In the sidebar, go to **Settings → Billing → Change plan**.  
2. Choose the plan you want to switch to and click **Con

C:\Users\sahil\AppData\Local\Temp\ipykernel_27368\1147667105.py:4: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='b328af90-325e-4312-914c-a851542bc6d1', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='6e5288d8-b8f3-47aa-b1db-dd95609b6032', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n